# fetch

> Download YouTube xscripts and metadata locally via yt-dlp

In [ ]:
#| default_exp fetch

## Storage design

**Storage location:** `~/.cache/yttoc/{video_id}/` (XDG Base Directory spec — re-downloadable data belongs in cache). Overridable via `--root`.

```
~/.cache/yttoc/
  {video_id}/
    meta.json
    captions.ja.srt        # if Japanese captions exist
    captions.en.srt        # otherwise fall back to English
```

**meta.json fields:**
```json
{
  "id": "kwSVtQ7dziU",
  "title": "Skill Issue: Andrej Karpathy on Code Agents...",
  "channel": "No Priors",
  "duration": 3991,
  "upload_date": "20260320",
  "webpage_url": "https://www.youtube.com/watch?v=kwSVtQ7dziU",
  "description": "Video description with possible manual ToC...",
  "captions": {"ja": "manual"},
  "last_used_at": "2026-04-06T14:22:31+00:00"
}
```

`description` is preserved for use as background context in ToC/summary generation. Many videos include a manual ToC with timestamps in their description.

**Caption selection logic:** try Japanese first (manual `ja`/`ja-*` → auto `ja`/`ja-*`), then English (manual `en`/`en-*` → auto `en`/`en-*`). Error if neither exists.

**CLI:** `yttoc-fetch <url>` fetches one video, prints video_id to stdout. Batch via shell: `cat urls.txt | xargs -I{} yttoc-fetch {}`

## Modularize

In [ ]:
#| export
import json, os
from datetime import datetime, timezone
from pathlib import Path
import yt_dlp
from yttoc.core import Meta

_DEFAULT_ROOT = Path(os.environ.get('XDG_CACHE_HOME', Path.home() / '.cache')) / 'yttoc'

In [ ]:
#| export
def _pick_lang(tracks: dict,
               base_lang: str = 'en' # Preferred base language
              ) -> str | None: # Best-matching language key
    "Select an exact or prefix language match from yt-dlp subtitle tracks."
    tracks = tracks or {}
    if base_lang in tracks: return base_lang
    for lang in sorted(k for k in tracks if k != 'live_chat'):
        if lang.startswith(f'{base_lang}-'):
            return lang
    return None

def _glob_srt(out_dir: str | Path, # Directory to search
              pattern: str = 'captions.*.srt' # Glob pattern
             ) -> list[Path]: # Sorted matching paths
    "Find SRT files matching a glob pattern."
    return sorted(Path(out_dir).glob(pattern))

def _update_last_used(meta_path: Path # Path to meta.json
                     ) -> None:
    "Update last_used_at timestamp in meta.json."
    meta = Meta.model_validate_json(meta_path.read_text(encoding='utf-8'))
    meta.last_used_at = datetime.now(timezone.utc)
    meta_path.write_text(meta.model_dump_json(indent=2), encoding='utf-8')

def _build_meta(info: dict, # yt-dlp info dict
                lang: str = 'en', # Language that was fetched
                caption_type: str = 'auto' # 'manual' or 'auto'
               ) -> Meta: # Parsed Meta instance
    "Extract fields for meta.json from yt-dlp info."
    meta = Meta(
        id=info['id'],
        title=info['title'],
        channel=info['channel'],
        duration=info['duration'],
        upload_date=info['upload_date'],
        webpage_url=info['webpage_url'],
        description=info.get('description', ''),
        captions={lang: caption_type},
        last_used_at=datetime.now(timezone.utc),
    )
    return meta

def _download_srt(url: str, info: dict, out_dir: Path
                 ) -> tuple[Path, str, str]: # (srt_path, lang, caption_type)
    "Download SRT captions in the video's original spoken language."
    base_lang = info.get('language')
    if not base_lang:
        raise ValueError(f"Cannot determine original language for {info['id']}")
    manual_lang = _pick_lang(info.get('subtitles', {}), base_lang)
    auto_lang = _pick_lang(info.get('automatic_captions', {}), base_lang)
    selected_lang = manual_lang or auto_lang
    if selected_lang is None:
        raise ValueError(f"No {base_lang} captions available for {info['id']}")

    sub_opt = 'writesubtitles' if manual_lang else 'writeautomaticsub'
    opts = {
        'skip_download': True, 'quiet': True,
        sub_opt: True,
        'subtitleslangs': [selected_lang],
        'subtitlesformat': 'srt',
        'outtmpl': str(out_dir / f'captions_{base_lang}.%(ext)s'),
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])

    srt_path = out_dir / f'captions.{base_lang}.srt'
    matches = _glob_srt(out_dir, f'captions_{base_lang}*.srt')
    if not matches:
        raise FileNotFoundError(f"yt-dlp did not write an srt caption for {info['id']}")
    if len(matches) > 1:
        raise ValueError(f"Ambiguous caption files: {', '.join(p.name for p in matches)}")
    if matches[0] != srt_path:
        matches[0].replace(srt_path)
    caption_type = 'manual' if manual_lang else 'auto'
    return srt_path, base_lang, caption_type

In [ ]:
# Test: helpers
assert _pick_lang({'en': []}) == 'en'
assert _pick_lang({'live_chat': [], 'en-US': []}) == 'en-US'
assert _pick_lang({'ja': []}) is None
assert _pick_lang({'ja': []}, 'ja') == 'ja'
assert _pick_lang({'ja-JP': []}, 'ja') == 'ja-JP'
assert _pick_lang({'en': []}, 'ja') is None

from tempfile import TemporaryDirectory

# _glob_srt with default pattern (cache lookup)
with TemporaryDirectory() as d:
    base = Path(d)
    assert _glob_srt(base) == []
    (base / 'captions.ja.srt').write_text('1', encoding='utf-8')
    assert _glob_srt(base) == [base / 'captions.ja.srt']

# _glob_srt with download prefix pattern
with TemporaryDirectory() as d:
    base = Path(d)
    sample = base / 'captions_en.en-US.srt'
    sample.write_text('1', encoding='utf-8')
    assert _glob_srt(base, 'captions_en*.srt') == [sample]

# Test: _build_meta
fake_info = {'id': 'X', 'title': 't', 'channel': 'c', 'duration': 1,
             'upload_date': '20260101', 'webpage_url': 'u',
             'description': 'desc', 'subtitles': {}, 'automatic_captions': {'en': []}}
m = _build_meta(fake_info, lang='en', caption_type='auto')
assert m.captions == {'en': 'auto'}
assert m.description == 'desc'

m_ja = _build_meta(fake_info, lang='ja', caption_type='manual')
assert m_ja.captions == {'ja': 'manual'}

# Test: _update_last_used
with TemporaryDirectory() as d:
    p = Path(d) / 'meta.json'
    p.write_text(json.dumps({
        'id': 'X', 'title': 't', 'channel': 'c', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://y.com/X',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }), encoding='utf-8')
    _update_last_used(p)
    updated = json.loads(p.read_text(encoding='utf-8'))
    assert updated['id'] == 'X'
    assert updated['last_used_at'] != '2000-01-01T00:00:00+00:00'
    from datetime import datetime
    assert datetime.fromisoformat(updated['last_used_at']).tzinfo is not None

print('ok')

In [ ]:
# Test: _download_srt language resolution (unit-level, no network)
# Picks the original spoken language from info['language'], preferring manual
# captions over auto. No cross-language fallback / translation.

def _resolve_lang(info):
    "Simulate _download_srt's language resolution without downloading."
    base = info.get('language')
    if not base:
        return None, None, None
    manual = _pick_lang(info.get('subtitles', {}), base)
    auto = _pick_lang(info.get('automatic_captions', {}), base)
    if manual or auto:
        return base, ('manual' if manual else 'auto'), (manual or auto)
    return None, None, None

# English video with manual en subs → en manual
lang, ctype, sel = _resolve_lang({'language': 'en', 'subtitles': {'en': []}, 'automatic_captions': {}})
assert (lang, ctype, sel) == ('en', 'manual', 'en')

# English video with only auto en → en auto
lang, ctype, sel = _resolve_lang({'language': 'en', 'subtitles': {}, 'automatic_captions': {'en': [], 'ja': []}})
assert (lang, ctype) == ('en', 'auto') and sel == 'en'

# Japanese video → picks ja, ignores en
lang, ctype, sel = _resolve_lang({'language': 'ja', 'subtitles': {}, 'automatic_captions': {'ja': [], 'en': []}})
assert (lang, ctype, sel) == ('ja', 'auto', 'ja')

# Original ja with ja-JP variant
lang, ctype, sel = _resolve_lang({'language': 'ja', 'subtitles': {'ja-JP': []}, 'automatic_captions': {}})
assert (lang, ctype, sel) == ('ja', 'manual', 'ja-JP')

# English video but no en captions at all → None (no cross-language fallback)
lang, ctype, sel = _resolve_lang({'language': 'en', 'subtitles': {}, 'automatic_captions': {'ja': []}})
assert lang is None

# Missing language field → None
lang, ctype, sel = _resolve_lang({'subtitles': {'en': []}, 'automatic_captions': {}})
assert lang is None

print('ok')

In [ ]:
#| export
def get_video_info(url: str # YouTube video URL
                  ) -> dict: # yt-dlp info dict
    "Extract video metadata and caption info without downloading."
    with yt_dlp.YoutubeDL({'skip_download': True, 'quiet': True}) as ydl:
        return ydl.extract_info(url, download=False)

In [ ]:
#| network
# Test: get_video_info smoke test (requires network)
info = get_video_info('https://www.youtube.com/watch?v=kwSVtQ7dziU')
for k in ['id', 'title', 'channel', 'duration', 'upload_date', 'webpage_url']:
    assert k in info, f'missing {k}'
assert info['id'] == 'kwSVtQ7dziU'
print('ok')

In [ ]:
#| export
def fetch_video(url: str, # YouTube video URL
                info: dict, # Result of get_video_info
                root: str | Path = None, # Root download directory (default: ~/.cache/yttoc)
               ) -> Path: # Path to video directory
    "Save metadata and srt captions for one video in its original spoken language."
    root = Path(root) if root else _DEFAULT_ROOT
    out_dir = root / info['id']
    out_dir.mkdir(parents=True, exist_ok=True)

    meta_path = out_dir / 'meta.json'
    if _glob_srt(out_dir) and meta_path.exists():
        _update_last_used(meta_path)
        return out_dir

    _srt_path, lang, caption_type = _download_srt(url, info, out_dir)
    meta = _build_meta(info, lang=lang, caption_type=caption_type)
    meta_path.write_text(meta.model_dump_json(indent=2), encoding='utf-8')
    return out_dir

In [ ]:
#| eval: false
# Test: fetch_video (requires network)
import shutil
test_root = '/tmp/yttoc_test'
shutil.rmtree(test_root, ignore_errors=True)

url = 'https://www.youtube.com/watch?v=kwSVtQ7dziU'
info = get_video_info(url)
out = fetch_video(url, info, root=test_root)
assert (out / 'meta.json').exists()
assert _glob_srt(out)

meta = json.loads((out / 'meta.json').read_text(encoding='utf-8'))
assert meta['id'] == 'kwSVtQ7dziU'
assert 'last_used_at' in meta
assert 'captions' in meta

shutil.rmtree(test_root)
print('ok')

## CLI

In [ ]:
#| export
from fastcore.script import call_parse

@call_parse
def yttoc_fetch(url: str, # YouTube video URL
                root: str = None, # Root download directory (default: ~/.cache/yttoc)
               ):
    "Fetch metadata and captions for a single YouTube video in its original spoken language."
    info = get_video_info(url)
    out = fetch_video(url, info, root)
    print(info['id'])

In [ ]:
#| export
from yttoc.core import fmt_duration as _fmt_duration

In [ ]:
# Test: _fmt_duration (imported from core)
assert _fmt_duration(3991) == '1:06:31'
assert _fmt_duration(195) == '3:15'
assert _fmt_duration(0) == '0:00'
print('ok')

In [ ]:
#| export
@call_parse
def yttoc_list(root: str = None, # Root directory (default: ~/.cache/yttoc)
              ):
    "List cached videos sorted by last used."
    root = Path(root) if root else _DEFAULT_ROOT
    if not root.exists(): return

    items = []  # list of (meta: Meta, langs: str)
    for d in root.iterdir():
        if not d.is_dir(): continue
        meta_path = d / 'meta.json'
        srt_files = _glob_srt(d)
        if not (meta_path.exists() and srt_files): continue
        meta = Meta.model_validate_json(meta_path.read_text(encoding='utf-8'))
        captions = meta.captions or {p.stem.split('.', 1)[1]: '?' for p in srt_files}
        langs = ','.join(sorted(captions.keys()))
        items.append((meta, langs))

    items.sort(key=lambda x: x[0].last_used_at, reverse=True)
    for meta, langs in items:
        ts = meta.last_used_at.isoformat()[:16].replace('T', ' ')
        dur = _fmt_duration(meta.duration)
        print(f"{meta.id}  {ts}  {dur:>8}  [{langs}]  {meta.title}")


In [ ]:
# Test: yttoc_list output
import io, contextlib
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    # Video A: older, English only
    a = root / 'AAAA'
    a.mkdir()
    (a / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (a / 'meta.json').write_text(json.dumps({
        'id': 'AAAA', 'title': 'Old video', 'channel': 'Ch', 'duration': 195,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=AAAA',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2026-01-01T00:00:00+00:00',
    }))
    # Video B: newer, Japanese
    b = root / 'BBBB'
    b.mkdir()
    (b / 'captions.ja.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nこんにちは\n')
    (b / 'meta.json').write_text(json.dumps({
        'id': 'BBBB', 'title': 'New video', 'channel': 'Ch', 'duration': 3991,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=BBBB',
        'description': '', 'captions': {'ja': 'manual'},
        'last_used_at': '2026-04-06T15:00:00+00:00',
    }))
    # Video C: incomplete (no srt) — should be skipped
    c = root / 'CCCC'
    c.mkdir()
    (c / 'meta.json').write_text('{"id":"CCCC"}')

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_list(root=str(root))
    out = buf.getvalue()

    lines = [l for l in out.strip().splitlines() if l.strip()]
    assert len(lines) == 2, f'expected 2 lines, got {len(lines)}'
    assert 'CCCC' not in out, 'incomplete video CCCC should be skipped'
    assert 'BBBB' in out, 'Japanese video should appear'
    assert '[ja]' in out, 'BBBB should show ja'
    assert '[en]' in out, 'AAAA should show en'
    # BBBB before AAAA (newer first)
    pos_b = out.index('BBBB')
    pos_a = out.index('AAAA')
    assert pos_b < pos_a, 'BBBB (newer) should appear before AAAA (older)'

# Legacy: yttoc_list handles meta.json with empty captions dict (falls back to filename detection)
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'LEGACY'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'LEGACY', 'title': 'Legacy video', 'channel': 'Ch', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=LEGACY',
        'description': '', 'captions': {},
        'last_used_at': '2026-01-01T00:00:00+00:00',
    }))
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_list(root=str(root))
    out = buf.getvalue()
    assert 'LEGACY' in out, 'legacy video should appear'
    assert '[en]' in out, 'should detect language from filename'

print('ok')

In [ ]:
# Test: yttoc_list rejects a corrupted meta.json (invalid caption type)
import io, contextlib
from tempfile import TemporaryDirectory
from pydantic import ValidationError

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'BAD1'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'BAD1', 'title': 'T', 'channel': 'C', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://y.com',
        'description': '', 'captions': {'en': 'autop'},
        'last_used_at': '2026-01-01T00:00:00+00:00'}))

    try:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            yttoc_list(root=str(root))
    except ValidationError:
        pass
    else:
        assert False, 'expected ValidationError for invalid caption type in meta.json'
print('ok')


In [ ]:
# Test: _update_last_used round-trip — bumps last_used_at monotonically, result is datetime-parseable
import time
from datetime import datetime
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    p = Path(d) / 'meta.json'
    p.write_text(json.dumps({
        'id': 'RT1', 'title': 't', 'channel': 'c', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://y.com',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }), encoding='utf-8')

    _update_last_used(p)
    first = json.loads(p.read_text(encoding='utf-8'))['last_used_at']
    assert first != '2000-01-01T00:00:00+00:00'
    first_dt = datetime.fromisoformat(first)
    assert first_dt.tzinfo is not None

    time.sleep(0.001)
    _update_last_used(p)
    second = json.loads(p.read_text(encoding='utf-8'))['last_used_at']
    second_dt = datetime.fromisoformat(second)
    assert second_dt >= first_dt, f'{second} should be >= {first}'
print('ok')


In [ ]:
# Test: fetch_video cache hit with existing captions
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    fake_info = {'id': 'TEST123', 'title': 't', 'channel': 'c',
                 'duration': 60, 'upload_date': '20260101',
                 'webpage_url': 'https://example.com', 'description': '',
                 'subtitles': {}, 'automatic_captions': {'en': [{'ext': 'srt'}]}}
    out_dir = Path(d) / 'TEST123'
    out_dir.mkdir()
    (out_dir / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\ntest\n')
    (out_dir / 'meta.json').write_text(json.dumps({
        'id': 'TEST123', 'title': 't', 'channel': 'c', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://example.com',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))
    result = fetch_video('https://example.com', fake_info, root=d)
    assert result.name == 'TEST123'
    meta = json.loads((result / 'meta.json').read_text())
    assert 'last_used_at' in meta

# Cache hit also works with ja srt
with TemporaryDirectory() as d:
    out_dir = Path(d) / 'TEST456'
    out_dir.mkdir()
    (out_dir / 'captions.ja.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nテスト\n')
    (out_dir / 'meta.json').write_text(json.dumps({
        'id': 'TEST456', 'title': 't', 'channel': 'c', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://example.com',
        'description': '', 'captions': {'ja': 'manual'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))
    result = fetch_video('https://example.com',
        {'id': 'TEST456', 'title': 't', 'channel': 'c', 'duration': 60,
         'upload_date': '20260101', 'webpage_url': 'u', 'description': '',
         'subtitles': {'ja': []}, 'automatic_captions': {}}, root=d)
    meta = json.loads((result / 'meta.json').read_text())
    assert meta['captions'] == {'ja': 'manual'}
print('ok')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()